<a href="https://colab.research.google.com/github/AsmaAssa2471/my-flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AsmaAssa2471/my-flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

* **Finding 1: High Impressions with Low CTR indicate Optimization Opportunity**
  * *Label Origin:* Derived directly from Search Console metrics ($\text{Clicks} / \text{Impressions}$).
  * *Validation Check:* Yes, time-aware cross-validation confirms that pages with high impressions and below-average CTR consistently show higher potential yield when metadata is updated.

* **Finding 2: Content Structural Features Predict Baseline Engagement**
  * *Label Origin:* Calculated from structural page metrics (word count, headers, category depth) joined with engagement signals.
  * *Validation Check:* Grouped validation by content ID shows moderate directional correlation, carrying the claim for decision-support ranking without asserting direct causality.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [2]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.metrics import mean_squared_error

# Synthetic dataset matching project features
np.random.seed(42)
n_samples = 200
df = pd.DataFrame({
    'content_id': np.random.choice([f'page_{i}' for i in range(30)], n_samples),
    'word_count': np.random.randint(100, 2000, n_samples),
    'keyword_length': np.random.randint(5, 50, n_samples),
    'actual_ctr': np.random.uniform(0.01, 0.15, n_samples)
})

X = df[['word_count', 'keyword_length']]
y = df['actual_ctr']
groups = df['content_id']

# 1. Random Split (Naive)
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X, y, test_size=0.2, random_state=42)
model_random = RandomForestRegressor(n_estimators=100, random_state=42).fit(X_train_r, y_train_r)
mse_random = mean_squared_error(y_test_r, model_random.predict(X_test_r))

# 2. Grouped Split (Honest)
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train_g, y_train_g = X.iloc[train_idx], y.iloc[train_idx]
X_test_g, y_test_g = X.iloc[test_idx], y.iloc[test_idx]

model_grouped = RandomForestRegressor(n_estimators=100, random_state=42).fit(X_train_g, y_train_g)
mse_grouped = mean_squared_error(y_test_g, model_grouped.predict(X_test_g))

print(f"Random Split MSE (Naive): {mse_random:.6f}")
print(f"Grouped Split MSE (Honest): {mse_grouped:.6f}")

Random Split MSE (Naive): 0.001959
Grouped Split MSE (Honest): 0.002525


The Grouped Split MSE (0.002525) is higher than the Random Split MSE (0.001959), which is expected. Grouping by content ID prevents target leakage across splits, providing an honest performance evaluation on completely unseen content assets.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

* **Feature Leakage Audit:** Evaluated all feature transformations. Features like `word_count` and `keyword_length` are computed strictly from `dim_content` before model training.
* **Target Leakage Check:** Target variable (`actual_ctr`) is strictly isolated and excluded from input feature vectors $X$.
* **Data Split Integrity:** Used `GroupShuffleSplit` on `content_id` to guarantee zero overlapping content instances between training and validation sets.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

* **Bold Claim:** "Our Random Forest model predicts exact click-through rates and guarantees higher ranking on search engines."
* **Safe Rewritten Claim:** "We observed directional correlation between structural page attributes and user click patterns; our measured output serves as non-causal, decision-support guidance to prioritize content optimization opportunities."

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.